# Training YOLO11 on Balanced Trash Dataset
7-class trash detection: battery, cardboard, paper, glass, metal, plastic, organic.
- Dataset: Trash_dataset_balanced (26,048 images: 18,320 train, 4,636 val, 3,092 test).
- Default model: YOLO11s (can be changed to yolo11n or yolo11m in cell 5).
- Checkpoints and logs are saved to Google Drive.

### 1. GPU Check & Setup

In [ ]:
!nvidia-smi
!pip install -q ultralytics pandas

### 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 3. Extract Dataset

In [ ]:
import os
import glob
import zipfile
zip_file = '/content/drive/MyDrive/Trash (3).zip'
if not os.path.exists(zip_file):
    zip_file = '/content/drive/MyDrive/trash (3).zip'

if zip_file and os.path.exists(zip_file):
    print(f'Extracting: {zip_file} to /content/trash_project...')
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall('/content/trash_project')
    print('Extraction complete.')
else:
    print('Error: Zip file not found on Google Drive.')
    print('Available zip files:', glob.glob('/content/drive/MyDrive/*.zip'))

### 4. Locate Balanced Dataset & Create data_balanced.yaml

In [ ]:
import os
import glob

dataset_root = '/content/trash_project/Trash/Trash_dataset_balanced'
if not os.path.exists(dataset_root):
    for root, dirs, files in os.walk('/content/trash_project'):
        if os.path.basename(root) == 'Trash_dataset_balanced':
            dataset_root = root
            break

print(f'Dataset path: {dataset_root}')

yaml_content = f"""
path: {dataset_root}
train: train/images
val: val/images
test: test/images

nc: 7
names:
  0: battery
  1: cardboard
  2: paper
  3: glass
  4: metal
  5: plastic
  6: organic
"""

with open('/content/data_balanced.yaml', 'w') as f:
    f.write(yaml_content.strip())


### 5. Training Model (Change model_name to 'yolo11n' or 'yolo11m' if needed)

In [ ]:
import os
import time
from ultralytics import YOLO

# Select model: 'yolo11s' (default), 'yolo11n', or 'yolo11m'
model_name = 'yolo11s'
weights = f'{model_name}.pt'
batch_size = 32
epochs = 100
patience = 10

drive_project = '/content/drive/MyDrive/Trash_YOLO11_Balanced_Runs'
exp_name = f'trash_{model_name}_balanced'
os.makedirs(drive_project, exist_ok=True)

def on_fit_epoch_end(trainer):
    if not hasattr(trainer, 'stopper') or trainer.stopper is None:
        return
    delta = trainer.epoch - trainer.stopper.best_epoch
    if delta == 5 and not getattr(trainer, '_lr_reduced_on_plateau', False):
        trainer._lr_reduced_on_plateau = True
        if hasattr(trainer, 'scheduler') and hasattr(trainer.scheduler, 'base_lrs'):
            trainer.scheduler.base_lrs = [b * 0.5 for b in trainer.scheduler.base_lrs]
        curr_lr = trainer.optimizer.param_groups[0]['lr'] if trainer.optimizer.param_groups else 0.0
        print('\n' + '-' * 75)
        print(f'[ReduceLROnPlateau] 5 epochs without improvement. Reducing learning rate to: {curr_lr:.6f}')
        print('-' * 75 + '\n')
    elif delta < 5:
        trainer._lr_reduced_on_plateau = False

print(f'Starting training: {model_name.upper()} (weights={weights}, batch={batch_size}, epochs={epochs})')
model = YOLO(weights)
model.add_callback('on_fit_epoch_end', on_fit_epoch_end)

t_start = time.time()
train_results = model.train(
    data='/content/data_balanced.yaml',
    epochs=epochs,
    batch=batch_size,
    imgsz=640,
    device=0,
    workers=2,
    optimizer='AdamW',
    lr0=0.001,
    patience=patience,
    cos_lr=True,
    weight_decay=0.0005,
    dropout=0.1,
    mixup=0.15,
    copy_paste=0.3,
    erasing=0.4,
    project=drive_project,
    name=exp_name,
    save=True,
    verbose=True
)
duration_min = round((time.time() - t_start) / 60, 1)
print(f'Training finished in {duration_min} minutes.')

# Evaluation on test set
best_weights_path = os.path.join(drive_project, exp_name, 'weights', 'best.pt')
print(f'\nEvaluating on test set: {best_weights_path}')
eval_model = YOLO(best_weights_path)
test_res = eval_model.val(
    data='/content/data_balanced.yaml',
    split='test',
    imgsz=640,
    device=0,
    verbose=True
)

params_count = round(sum(p.numel() for p in eval_model.model.parameters()) / 1e6, 2)
print('\n' + '-' * 65)
print(f'Results for {model_name.upper()} on Test Set:')
print('-' * 65)
print(f'  Parameters : {params_count}M')
print(f'  Precision  : {test_res.box.mp:.4f} ({test_res.box.mp*100:.2f}%)')
print(f'  Recall     : {test_res.box.mr:.4f} ({test_res.box.mr*100:.2f}%)')
print(f'  mAP@50     : {test_res.box.map50:.4f} ({test_res.box.map50*100:.2f}%)')
print(f'  mAP@50-95  : {test_res.box.map:.4f} ({test_res.box.map*100:.2f}%)')
print(f'  Latency    : {test_res.speed.get("inference", 0.0):.2f} ms')
print('-' * 65)

print('\nPer-class mAP50-95:')
for i, c in enumerate(test_res.names.values()):
    print(f'  {c:<12}: {test_res.box.maps[i]:.4f}')